In [1]:
!pip install -q PyPDF2
import os
import zipfile
import shutil
from google.colab import files
import PyPDF2
from IPython.display import display, HTML
print(f"PyPDF2 version: {PyPDF2.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 3.2 MB/s eta 0:00:00
PyPDF2 version: 3.0.1


In [2]:
def compress_and_zip_pdfs(directory_path=None, quality_level='medium'):
    pdf_files_to_process = []
    temp_dir = 'temp_compressed_pdfs'
    output_zip_name = 'compressed_pdfs.zip'

    # Create a temporary directory for compressed files
    os.makedirs(temp_dir, exist_ok=True)

    if directory_path:
        # Check if the provided directory is valid
        if not os.path.isdir(directory_path):
            print(f"Error: Directory '{directory_path}' not found or is not a valid directory. Please upload PDF files manually.")
            directory_path = None # Fallback to upload mechanism
        else:
            # Collect PDF files from the directory
            for filename in os.listdir(directory_path):
                if filename.lower().endswith('.pdf'):
                    pdf_files_to_process.append(os.path.join(directory_path, filename))
            if not pdf_files_to_process:
                print(f"No PDF files found in '{directory_path}'. Please upload PDF files manually.")
                directory_path = None # Fallback to upload mechanism

    if not directory_path:
        # If no valid directory, ask the user to upload files
        print("Please upload your PDF files. Only PDF files will be processed.")
        uploaded = files.upload()
        for filename, content in uploaded.items():
            if filename.lower().endswith('.pdf'):
                with open(filename, 'wb') as f:
                    f.write(content)
                pdf_files_to_process.append(filename)
            else:
                print(f"Skipping non-PDF file: {filename}")

    if not pdf_files_to_process:
        print("No PDF files to process. Exiting.")
        shutil.rmtree(temp_dir) # Clean up temporary directory
        return

    compressed_files = []
    print(f"Found {len(pdf_files_to_process)} PDF files to process...")

    for pdf_file_path in pdf_files_to_process:
        try:
            input_pdf = PyPDF2.PdfReader(pdf_file_path)
            output_pdf_path = os.path.join(temp_dir, f"compressed_{os.path.basename(pdf_file_path)}")
            output_pdf = PyPDF2.PdfWriter()

            # Iterate over pages and add them to the writer
            for page_num in range(len(input_pdf.pages)):
                page = input_pdf.pages[page_num]
                output_pdf.add_page(page)

            # Apply compression based on quality_level
            if quality_level == 'high':
                output_pdf.compress_content_streams()
                print(f"Applying high compression for: {os.path.basename(pdf_file_path)}")
            elif quality_level == 'medium':
                # Default PyPDF2 write might still offer some compression
                print(f"Applying medium compression for: {os.path.basename(pdf_file_path)}")
            elif quality_level == 'low':
                # For 'low', we don't explicitly call compress_content_streams
                # PyPDF2's default write operation might still optimize somewhat
                print(f"Applying low compression for: {os.path.basename(pdf_file_path)}")
            else:
                print(f"Invalid quality_level '{quality_level}'. Using default medium compression for: {os.path.basename(pdf_file_path)}")

            # Save the compressed PDF
            with open(output_pdf_path, 'wb') as f_out:
                output_pdf.write(f_out)

            compressed_files.append(output_pdf_path)
            print(f"Successfully processed: {os.path.basename(pdf_file_path)}")
        except Exception as e:
            print(f"Error processing {os.path.basename(pdf_file_path)}: {e}")

    if not compressed_files:
        print("No files were successfully compressed. Exiting.")
        shutil.rmtree(temp_dir) # Clean up temporary directory
        return

    # Zip all compressed files
    with zipfile.ZipFile(output_zip_name, 'w') as zipf:
        for file_path in compressed_files:
            zipf.write(file_path, os.path.basename(file_path))
    print(f"All compressed PDFs zipped into '{output_zip_name}'.")

    # Download the zip file
    files.download(output_zip_name)
    print(f"'{output_zip_name}' downloaded to your current directory.")

    # Clean up temporary directory and uploaded files (if any)
    shutil.rmtree(temp_dir)
    if not directory_path: # Clean up files uploaded to the root if no directory was used
        for f in pdf_files_to_process:
            try:
                os.remove(f)
            except OSError:
                pass # File might have been cleaned up by another mechanism
    print("Cleanup complete.")

The following function `compress_and_zip_pdfs` takes an optional `directory_path` argument. If provided and valid, it will process PDF files from that directory. Otherwise, it will prompt you to upload PDF files directly. It then compresses these PDFs, zips them into `compressed_pdfs.zip`, and downloads the zip file.

### How to use the function:

The `compress_and_zip_pdfs` function now accepts an optional `quality_level` parameter, which can be `'high'`, `'medium'`, or `'low'`. By default, it's set to `'medium'`.

- **'high'**: Applies `PyPDF2.PdfWriter().compress_content_streams()` for potentially greater compression.
- **'medium'**: Uses `PyPDF2`'s default writing, which might still offer some compression.
- **'low'**: Similar to 'medium', no explicit compression method is called, relying on default `PyPDF2` behavior.

**Option 1: Process PDFs from a directory (if you have mounted Google Drive or created a directory with PDFs):**

First, you might need to mount your Google Drive if your PDFs are there:

```python
from google.colab import drive
drive.mount('/content/drive')
```

Then, call the function with the path to your directory and desired quality level:

```python
# Example: If your PDFs are in '/content/drive/MyDrive/my_pdfs'
# compress_and_zip_pdfs(directory_path='/content/drive/MyDrive/my_pdfs', quality_level='high')
# compress_and_zip_pdfs(directory_path='/content/drive/MyDrive/my_pdfs', quality_level='medium')
# compress_and_zip_pdfs(directory_path='/content/drive/MyDrive/my_pdfs') # Uses default 'medium'
```

**Option 2: Upload PDFs manually:**

If you don't specify a directory or the directory is invalid, the function will prompt you to upload files directly from your computer.

```python
# This will open a file upload dialog in your browser
# compress_and_zip_pdfs(quality_level='high')
# compress_and_zip_pdfs() # Uses default 'medium'
```

**Note:** This function uses `PyPDF2`, which performs a basic compression by rewriting the PDF. The level of compression may vary depending on the original PDF content. For more advanced compression, specialized libraries or tools might be needed.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
compress_and_zip_pdfs('/content/drive/MyDrive/00_AI.Garden/01_Github/600_AI-Sys/L7_Apps/Investment/futureTech', quality_level='high')

Found 12 PDF files to process...
Error processing 260430.01-Quantum Computing and the “TSMC” for QPU (The Future of Tech) (BlackBook BERN_254296).pdf: 'PdfWriter' object has no attribute 'compress_content_streams'
Error processing 260430.02-Physical AI — Bridging Industrial and Humanoid Robotics (The Future of Tech) (BlackBook BERN_254296).pdf: 'PdfWriter' object has no attribute 'compress_content_streams'
Error processing 260430.03-AI Datacenter Networking (The Future of Tech) (BlackBook BERN_254296).pdf: 'PdfWriter' object has no attribute 'compress_content_streams'
Error processing 260430.04-Mapping the CPO Value Chain (The Future of Tech) (BlackBook BERN_254296).pdf: 'PdfWriter' object has no attribute 'compress_content_streams'
Error processing 260430.05-BCI-Brain-Computer-Interface_Concept to Reality (The Future of Tech) (BlackBook BERN_254296).pdf: 'PdfWriter' object has no attribute 'compress_content_streams'
Error processing 260430.06-Debating the TAM for Agentic AI Revenue (T